In [1]:
from pathlib import Path
import json, shutil, datetime, platform, sys
import numpy as np
import pandas as pd
RUNS_ROOT = Path.home() / "rr_runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def _ts():
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def make_run_dir(method: str, dataset: str, tag: str = "") -> Path:
    tag = f"_{tag}" if tag else ""
    run_dir = RUNS_ROOT / method / dataset / f"{_ts()}{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def env_info():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }

def save_run(run_dir: Path,
             per_window_df: pd.DataFrame,
             per_subject_df: pd.DataFrame,
             config: dict,
             splits_path: str | None = None):
    per_window_path = run_dir / "per_window.csv"
    per_subject_path = run_dir / "per_subject.csv"
    config_path = run_dir / "config.json"

    per_window_df.to_csv(per_window_path, index=False)
    per_subject_df.to_csv(per_subject_path, index=False)

    config = dict(config)
    config["environment"] = env_info()
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    if splits_path is not None:
        sp = Path(splits_path)
        if sp.exists():
            shutil.copy2(sp, run_dir / sp.name)

    print("\n✅ Saved run to:", str(run_dir))
    print("  -", per_window_path.name, "| rows:", len(per_window_df))
    print("  -", per_subject_path.name, "| rows:", len(per_subject_df))
    print("  -", config_path.name)
    if splits_path is not None:
        print("  - splits:", Path(splits_path).name, ("OK" if Path(splits_path).exists() else "MISSING"))


In [2]:
import os, json, gc, pickle, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.decomposition import FastICA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler


In [24]:
import os, random
import numpy as np
import torch

def seed_everything_deterministic(seed=0):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism flags (GPU)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional: strict determinism (may error if an op has no deterministic impl)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("⚠️ Could not enable strict deterministic algorithms:", e)

seed_everything_deterministic(123)


In [3]:
# ====== EDIT THESE PATHS ======
PPG_DALIA_RAW_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Raw_Signal.pkl"
PPG_DALIA_ANN_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Annotation.pkl"
WESAD_RAW_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Raw_Signal.pkl"
WESAD_ANN_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Annotation.pkl"

# where your saved splits actually are (baseline step)

import os, json, shutil

OUT_DIR = r"C:\Users\yasmi\OneDrive\Документы\kazemi_fusion_out"
os.makedirs(OUT_DIR, exist_ok=True)

BASELINE_OUT_DIR = r"C:\Jupyter Files\baseline_outputs"
SPLITS_JSON = os.path.join(BASELINE_OUT_DIR, "splits_loso.json")


print("SPLITS_JSON:", SPLITS_JSON)
print("exists?:", os.path.exists(SPLITS_JSON))
assert os.path.exists(SPLITS_JSON), "Splits JSON path is wrong"
with open(SPLITS_JSON, "r", encoding="utf-8") as f:
    splits = json.load(f)

print("Loaded splits keys:", list(splits.keys()))

# ====== Fixed window assumptions (your assets already satisfy this) ======
FS = 64.0
WIN_SAMPLES = 2048

# ====== Channel mapping (based on your sweep) ======
# PPG channel indices:
WESAD_PPG_CH = 1
DALIA_PPG_CH = 0

# IMU channels:
# Assumption: remaining channels after PPG are [ACCx, ACCy, ACCz, GYRx, GYRy, GYRz] if present.
# Your WESAD raw has 4 channels total -> likely PPG + 3-axis ACC (no gyro).
WESAD_ACC_CHS = [0, 2, 3]
WESAD_GYR_CHS = None  # no gyro

# Your DaLiA raw has 7 channels total -> likely PPG + 3-axis ACC + 3-axis GYR.
DALIA_ACC_CHS = [1, 2, 3]
DALIA_GYR_CHS = [4, 5, 6]


SPLITS_JSON: C:\Jupyter Files\baseline_outputs\splits_loso.json
exists?: True
Loaded splits keys: ['protocol', 'WESAD_subjects', 'PPG_DaLiA_subjects', 'folds_WESAD', 'folds_PPG_DaLiA']


In [4]:
def load_pkl(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, "rb") as f:
        return pickle.load(f)

def load_kazemi_raw(pkl_path: str) -> np.ndarray:
    """Returns raw windows as np.ndarray (N,C,L) or (N,L)."""
    obj = load_pkl(pkl_path)
    X = np.asarray(obj)
    return X

def load_kazemi_annotation_df(pkl_path: str) -> pd.DataFrame:
    obj = load_pkl(pkl_path)
    assert isinstance(obj, pd.DataFrame), f"Expected DataFrame, got {type(obj)}"
    return obj

def get_rr_sid_from_ann(df: pd.DataFrame):
    # Your df columns are: ['Reference_RR','activity_id','patient_id']
    rr = df["Reference_RR"].to_numpy(dtype=np.float32)
    sid = df["patient_id"].to_numpy(dtype=int)
    return rr, sid

def windowing_checks(X, rr, sid, name):
    print(f"\n=== {name} windowing checks ===")
    print("X shape:", X.shape)
    print("rr shape:", rr.shape, "sid shape:", sid.shape)
    assert X.shape[-1] == WIN_SAMPLES, f"{name}: expected L={WIN_SAMPLES}"
    assert len(rr) == X.shape[0] == len(sid), f"{name}: X/rr/sid mismatch"
    print("unique subjects:", np.unique(sid))
    print("✅ PASS")


In [5]:
wesad_raw = load_kazemi_raw(WESAD_RAW_PKL)
dalia_raw = load_kazemi_raw(PPG_DALIA_RAW_PKL)

wesad_ann_df = load_kazemi_annotation_df(WESAD_ANN_PKL)
dalia_ann_df = load_kazemi_annotation_df(PPG_DALIA_ANN_PKL)

wesad_rr, wesad_sid = get_rr_sid_from_ann(wesad_ann_df)
dalia_rr, dalia_sid = get_rr_sid_from_ann(dalia_ann_df)

windowing_checks(wesad_raw, wesad_rr, wesad_sid, "WESAD")
windowing_checks(dalia_raw, dalia_rr, dalia_sid, "PPG-DaLiA")



=== WESAD windowing checks ===
X shape: (1797, 4, 2048)
rr shape: (1797,) sid shape: (1797,)
unique subjects: [ 2  3  4  5  6  7  8  9 10 11]
✅ PASS

=== PPG-DaLiA windowing checks ===
X shape: (3883, 7, 2048)
rr shape: (3883,) sid shape: (3883,)
unique subjects: [ 1  2  3  4  5  7  8  9 10 11 12 13 14 15]
✅ PASS


In [6]:
from sklearn.exceptions import ConvergenceWarning
import warnings

def pick_ica_resp(sig_3axis: np.ndarray, fs: float) -> np.ndarray:
    """
    Returns one respiration-like component from 3-axis IMU.
    If ICA fails/non-converges, falls back to first principal direction (SVD).
    """
    X = sig_3axis.T.astype(np.float64)  # (L,3)
    X = X - X.mean(axis=0, keepdims=True)

    comps = None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        try:
            ica = FastICA(
                n_components=3,
                random_state=0,
                max_iter=1000,     # more robust
                tol=1e-3,          # slightly looser
                whiten="unit-variance"
            )
            comps = ica.fit_transform(X)  # (L,3)
        except Exception:
            comps = None

    if comps is None or not np.all(np.isfinite(comps)):
        # Fallback: SVD first component (always stable)
        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        resp = U[:, 0]
    else:
        # choose component with strongest resp-band energy
        freqs = np.fft.rfftfreq(comps.shape[0], d=1/fs)
        band = (freqs >= 0.10) & (freqs <= 0.70)

        best_idx, best_pow = 0, -1.0
        for k in range(3):
            spec = np.abs(np.fft.rfft(comps[:, k]))**2
            p = float(spec[band].sum())
            if p > best_pow:
                best_pow = p
                best_idx = k
        resp = comps[:, best_idx]

    return resp.astype(np.float32)


def minmax_to_minus1_1(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    return (2.0 * (x - mn) / (mx - mn + eps) - 1.0).astype(np.float32)

def build_fusion_inputs(raw: np.ndarray, ppg_ch: int, acc_chs, gyr_chs, fs: float) -> np.ndarray:
    """
    raw: (N,C,L)
    returns: fusion_X (N,3,L) = [PPG, Resp(ACC), Resp(GYR)]
    """
    raw = np.asarray(raw)
    assert raw.ndim == 3, f"Expected (N,C,L), got {raw.shape}"
    N, C, L = raw.shape

    ppg = raw[:, ppg_ch, :].astype(np.float32)

    acc = raw[:, acc_chs, :].astype(np.float32)  # (N,3,L)
    if gyr_chs is None:
        gyr = None
    else:
        gyr = raw[:, gyr_chs, :].astype(np.float32)  # (N,3,L)

    fusion = np.zeros((N, 3, L), dtype=np.float32)

    for i in tqdm(range(N), desc="ICA extract"):
        resp_acc = pick_ica_resp(acc[i], fs)
        if gyr is None:
            resp_gyr = resp_acc  # no gyro available -> duplicate
        else:
            resp_gyr = pick_ica_resp(gyr[i], fs)

        # normalize each channel to [-1,1] like the paper
        fusion[i, 0, :] = minmax_to_minus1_1(ppg[i])
        fusion[i, 1, :] = minmax_to_minus1_1(resp_acc)
        fusion[i, 2, :] = minmax_to_minus1_1(resp_gyr)

    return fusion


In [7]:
wesad_X_fusion = build_fusion_inputs(wesad_raw, WESAD_PPG_CH, WESAD_ACC_CHS, WESAD_GYR_CHS, FS)
dalia_X_fusion = build_fusion_inputs(dalia_raw, DALIA_PPG_CH, DALIA_ACC_CHS, DALIA_GYR_CHS, FS)

print("WESAD fusion:", wesad_X_fusion.shape)
print("DaLiA fusion:", dalia_X_fusion.shape)

# free big raw arrays if needed
del wesad_raw, dalia_raw
gc.collect()


ICA extract: 100%|██████████| 3883/3883 [01:42<00:00, 37.93it/s] 


WESAD fusion: (1797, 3, 2048)
DaLiA fusion: (3883, 3, 2048)


29

In [8]:
# =========================
# Compute SQI01 from PPG windows (for coverage / stratification)
# =========================
import numpy as np

def compute_sqi01_from_ppg_windows(X_fusion, fs_hz, resp_band=(0.10, 0.70), ref_band=(0.10, 3.00), eps=1e-8):
    """
    SQI01 in [0,1]: fraction of spectral power in respiration band vs reference band.
    X_fusion: (N, C, L) where channel 0 is PPG.
    """
    ppg = X_fusion[:, 0, :].astype(np.float32)  # (N,L)
    N, L = ppg.shape

    # window + FFT
    win = np.hanning(L).astype(np.float32)
    xw = ppg * win[None, :]

    # rFFT power
    Xf = np.fft.rfft(xw, axis=1)
    Pxx = (np.abs(Xf) ** 2).astype(np.float32)  # (N, F)
    freqs = np.fft.rfftfreq(L, d=1.0/fs_hz).astype(np.float32)

    # masks
    rb = (freqs >= resp_band[0]) & (freqs <= resp_band[1])
    fb = (freqs >= ref_band[0]) & (freqs <= ref_band[1])

    # band powers
    resp_pow = np.sum(Pxx[:, rb], axis=1)
    ref_pow  = np.sum(Pxx[:, fb], axis=1)

    sqi = resp_pow / (ref_pow + eps)
    sqi = np.clip(sqi, 0.0, 1.0)  # keep it bounded for stability
    return sqi.astype(np.float32)

wesad_sqi_raw = compute_sqi01_from_ppg_windows(wesad_X_fusion, FS)
dalia_sqi_raw = compute_sqi01_from_ppg_windows(dalia_X_fusion, FS)

print("WESAD SQI01:", "finite", float(np.mean(np.isfinite(wesad_sqi_raw))),
      "min", float(np.min(wesad_sqi_raw)), "median", float(np.median(wesad_sqi_raw)), "max", float(np.max(wesad_sqi_raw)))
print("DaLiA SQI01:", "finite", float(np.mean(np.isfinite(dalia_sqi_raw))),
      "min", float(np.min(dalia_sqi_raw)), "median", float(np.median(dalia_sqi_raw)), "max", float(np.max(dalia_sqi_raw)))


WESAD SQI01: finite 1.0 min 0.01772656850516796 median 0.7043289542198181 max 0.9894029498100281
DaLiA SQI01: finite 1.0 min 0.0005253646522760391 median 0.08641990274190903 max 0.9372381567955017


In [9]:
def normalize_splits(splits, dataset_name: str, sid: np.ndarray):
    """
    Returns a dict-of-folds that run_loso() expects.
    Supports:
      A) already dict-of-folds: {"2":{"train":[...],"test":[2]}, ...}
      B) list of test subjects: [2,3,4,...] (we build train from all subjects)
    """
    sid_all = list(np.unique(np.asarray(sid).astype(int)))

    # choose the correct key from your JSON
    if dataset_name.lower() == "wesad":
        key = "folds_WESAD"
    else:
        key = "folds_PPG_DaLiA"

    folds = splits[key]

    # Case A: already dict-of-folds
    if isinstance(folds, dict):
        return folds

    # Case B: list of test subject ids
    if isinstance(folds, list):
        out = {}
        for s in folds:
            s = int(s)
            train = [x for x in sid_all if x != s]
            out[str(s)] = {"train": train, "test": [s]}
        return out

    raise ValueError(f"Unsupported folds type: {type(folds)}")


In [10]:
import numpy as np
import torch
from torch.utils.data import Dataset

class WindowDataset(Dataset):
    def __init__(self, X, y, sid, keep_subjects, max_windows=None, seed=0):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.float32).reshape(-1)
        self.sid = np.asarray(sid, dtype=int).reshape(-1)

        keep_subjects = np.asarray(keep_subjects, dtype=int)
        idx = np.where(np.isin(self.sid, keep_subjects))[0]

        # optional deterministic subsample (debug speed)
        if max_windows is not None and len(idx) > max_windows:
            rng = np.random.default_rng(seed)
            idx = rng.choice(idx, size=max_windows, replace=False)

        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = int(self.idx[k])
        x = torch.tensor(self.X[i], dtype=torch.float32)   # (C,L)
        y = torch.tensor(self.y[i], dtype=torch.float32)   # scalar
        sid = torch.tensor(int(self.sid[i]), dtype=torch.int64)
        win_idx = torch.tensor(i, dtype=torch.int64)
        return x, y, sid, win_idx


In [11]:
import torch
import torch.nn as nn

class ResidualInceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, slope=0.2):
        super().__init__()
        b = out_ch // 3
        b1, b2, b3 = b, b, out_ch - 2*b

        self.br1 = nn.Conv1d(in_ch, b1, 3, stride=2, padding=1, dilation=1, bias=False)
        self.br2 = nn.Conv1d(in_ch, b2, 3, stride=2, padding=2, dilation=2, bias=False)
        self.br3 = nn.Conv1d(in_ch, b3, 3, stride=2, padding=4, dilation=4, bias=False)

        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.LeakyReLU(slope, inplace=True)
        self.res = nn.Conv1d(in_ch, out_ch, 1, stride=2, padding=0, bias=False)

    def forward(self, x):
        y = torch.cat([self.br1(x), self.br2(x), self.br3(x)], dim=1)
        y = self.act(self.bn(y))
        return y + self.res(x)

class KazemiRRNet(nn.Module):
    def __init__(self, in_ch=3, slope=0.2):
        super().__init__()
        filters = [8, 16, 32, 64, 128, 256, 512, 1024]
        blocks = []
        ch = in_ch
        for f in filters:
            blocks.append(ResidualInceptionBlock(ch, f, slope=slope))
            ch = f
        self.backbone = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(ch, 64)
        self.fc2 = nn.Linear(64, 1)
        self.act = nn.LeakyReLU(slope, inplace=True)

    def forward(self, x):
        z = self.backbone(x)
        z = self.gap(z).squeeze(-1)
        z = self.act(self.fc1(z))
        return self.fc2(z).squeeze(-1)


In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler

def train_one_fold(
    X, y, sid,
    fold_train_subjects, fold_test_subjects,
    device,
    val_subject=None,             # ✅ NEW: subject-wise validation
    seed=123,
    epochs=60,
    steps_per_epoch=120,
    batch_size=64,
    lr=5e-4,
    patience=10,
    weight_decay=1e-4,
    max_train_windows=None,
    max_test_windows=None,
    print_every=1,
):
    """
    Trains on fold_train_subjects, validates on val_subject (a subject id from train),
    tests on fold_test_subjects (1 subject).
    """

    fold_train_subjects = [int(s) for s in fold_train_subjects]
    fold_test_subjects  = [int(s) for s in fold_test_subjects]
    assert len(fold_test_subjects) == 1, "Expected exactly 1 test subject in LOSO."

    # pick a validation subject if not provided
    if val_subject is None:
        rng = np.random.RandomState(seed + 2024)
        val_subject = int(rng.choice(fold_train_subjects))

    # train subjects exclude the val subject
    train_subjects = [s for s in fold_train_subjects if s != int(val_subject)]
    val_subject = int(val_subject)

    print(f"train_one_fold: train_subs={len(train_subjects)} "
          f"val_sub=1 test_subs={len(fold_test_subjects)}", flush=True)

    torch.manual_seed(seed)
    np.random.seed(seed)

    # --- datasets ---
    ds_train = WindowDataset(
        X, y, sid,
        keep_subjects=train_subjects,
        max_windows=max_train_windows
    )
    ds_val = WindowDataset(
        X, y, sid,
        keep_subjects=[val_subject],
        max_windows=max_test_windows
    )
    ds_test = WindowDataset(
        X, y, sid,
        keep_subjects=fold_test_subjects,
        max_windows=max_test_windows
    )

    # --- dataloaders ---
    sampler = RandomSampler(ds_train, replacement=True, num_samples=steps_per_epoch * batch_size)
    dl_train = DataLoader(ds_train, batch_size=batch_size, sampler=sampler, drop_last=True)
    dl_val   = DataLoader(ds_val,   batch_size=256, shuffle=False)
    dl_test  = DataLoader(ds_test,  batch_size=256, shuffle=False)

    # --- model ---
    model = KazemiRRNet(in_ch=X.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.SmoothL1Loss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for ep in range(epochs):
        model.train()
        for xb, yb, sidb, idxb in dl_train:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb).view(-1)
            loss = loss_fn(pred, yb.view(-1))

            opt.zero_grad()
            loss.backward()
            opt.step()

        sched.step()

        # --- validate on held-out subject ---
        model.eval()
        with torch.no_grad():
            v_abs = []
            for xb, yb, sidb, idxb in dl_val:
                xb = xb.to(device)
                pred = model(xb).view(-1).cpu().numpy()
                ytrue = yb.numpy().reshape(-1)
                v_abs.append(np.abs(pred - ytrue))
            val_mae = float(np.mean(np.concatenate(v_abs))) if len(v_abs) else float("inf")

        if (ep % print_every) == 0:
            print(f"  epoch {ep+1}/{epochs} | val_mae={val_mae:.3f} (val_subject={val_subject})", flush=True)

        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # --- test predictions (per-window) ---
    model.eval()
    rows = []
    with torch.no_grad():
        for xb, yb, sidb, idxb in dl_test:
            xb = xb.to(device)
            pred = model(xb).view(-1).cpu().numpy()
            ytrue = yb.numpy().reshape(-1)
            sid_np = sidb.numpy().reshape(-1)
            idx_np = idxb.numpy().reshape(-1)

            for i in range(len(pred)):
                rows.append({
                    "window_index": int(idx_np[i]),
                    "subject_id": int(sid_np[i]),
                    "rr_ref": float(ytrue[i]),
                    "rr_pred": float(pred[i]),
                    "is_valid": True
                })

    df = pd.DataFrame(rows)
    mae = float(np.mean(np.abs(df["rr_pred"].values - df["rr_ref"].values)))
    rmse = float(np.sqrt(np.mean((df["rr_pred"].values - df["rr_ref"].values) ** 2)))

    return df, {
        "MAE": mae,
        "RMSE": rmse,
        "N": int(len(df)),
        "best_val_mae": float(best_val),
        "val_subject": int(val_subject)
    }


In [13]:
# =========================
# Deterministic LOSO fold iterator + deterministic VAL selection
# =========================
import numpy as np
import pandas as pd
import time

def iter_folds_from_splits_anyformat(splits: dict, dataset_name: str):
    """
    Supports your splits keys:
      - folds_WESAD
      - folds_PPG_DaLiA
    And fold dict formats:
      - {"train_subjects":[...], "test_subjects":[...]}
      - {"train":[...], "test":[...]}
    """
    if dataset_name.lower().startswith("wesad"):
        key = "folds_WESAD"
    else:
        key = "folds_PPG_DaLiA"

    if key not in splits:
        raise KeyError(f"Missing '{key}' in splits. Available keys: {list(splits.keys())}")

    folds_obj = splits[key]
    if not isinstance(folds_obj, list) or len(folds_obj) == 0:
        raise ValueError(f"Bad splits format: splits['{key}'] must be a non-empty list")

    for fold in folds_obj:
        if "train_subjects" in fold and "test_subjects" in fold:
            train = [int(s) for s in fold["train_subjects"]]
            test  = [int(s) for s in fold["test_subjects"]]
        elif "train" in fold and "test" in fold:
            train = [int(s) for s in fold["train"]]
            test  = [int(s) for s in fold["test"]]
        else:
            raise ValueError(f"Bad fold keys: {fold.keys()}")
        yield {"train": train, "test": test}

def pick_val_subject_deterministic(train_subjects):
    """Deterministic validation subject: smallest subject id in training."""
    train_subjects = [int(s) for s in train_subjects]
    if len(train_subjects) < 2:
        raise ValueError("Need at least 2 training subjects to pick a val subject.")
    return int(np.min(train_subjects))


In [14]:
import torch, numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
torch.backends.cudnn.benchmark = True


Device: cpu


In [15]:
import pandas as pd
import numpy as np

def run_loso(dataset_name, X, y, sid, splits,
             device,
             epochs=60,
             steps_per_epoch=30,
             batch_size=64,
             patience=8,
             lr=1e-3,
             seed=0,
             max_train_windows=None,
             max_test_windows=None,
             print_every=1):
    """
    LOSO:
      - For each fold: pick ONE validation subject from the training subjects (subject-wise val)
      - Train on remaining train subjects
      - Test on held-out subject
    Returns:
      df_all: per-window predictions across all folds
      df_sub: per-subject metrics (one row per test subject)
    """
    all_rows = []
    per_subject = []

    folds = list(iter_folds_from_splits(splits, dataset_name))

    for fold_i, fold in enumerate(folds):
        train_subs_full = [int(s) for s in fold["train"]]
        test_subs       = [int(s) for s in fold["test"]]
        assert len(test_subs) == 1, "Expected 1 test subject per fold"

        # --- choose a validation subject from training subjects (deterministic) ---
        rng = np.random.RandomState(seed + 1337 + fold_i)
        val_subject = int(rng.choice(train_subs_full))
        train_subs  = [s for s in train_subs_full if s != val_subject]

        print(f"\n[{dataset_name}] fold {fold_i+1}/{len(folds)} | "
              f"test={test_subs[0]} | val={val_subject} | train_n={len(train_subs)}", flush=True)

        # train_one_fold MUST accept val_subject (see note below)
        df_fold, metrics = train_one_fold(
            X, y, sid,
            fold_train_subjects=train_subs,
            fold_test_subjects=test_subs,
            device=device,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            batch_size=batch_size,
            patience=patience,
            lr=lr,
            max_train_windows=max_train_windows,
            max_test_windows=max_test_windows,
            print_every=print_every,
            val_subject=val_subject
        )

        df_fold["dataset"] = dataset_name
        all_rows.append(df_fold)

        metrics["subject_id"]  = int(test_subs[0])
        metrics["val_subject"] = int(val_subject)
        per_subject.append(metrics)

        print(dataset_name, "test subject", test_subs[0], "|", metrics, flush=True)

    df_all = pd.concat(all_rows, ignore_index=True) if len(all_rows) else pd.DataFrame()
    df_sub = pd.DataFrame(per_subject).sort_values("subject_id") if len(per_subject) else pd.DataFrame()
    return df_all, df_sub


In [18]:
# =========================
# KAZEMI STRONG — FULL LOSO (deterministic val)
# =========================
def _call_kazemi_train_one_fold(train_one_fold_fn, X, y, sid, train_subs, val_sub, test_subs,
                               device, epochs, steps_per_epoch, batch_size, lr, weight_decay, patience, seed, print_every=1):
    """
    Tries multiple argument name styles to survive notebook drift.
    Expects to return either:
      - (df_test, metrics_dict)
      - metrics_dict containing y_true/y_pred/idx (less common)
    """
    # Most common (your logs): train_one_fold(X, y, sid, train_subs=..., val_sub=..., test_subs=..., ...)
    try:
        return train_one_fold_fn(
            X, y, sid,
            train_subs=train_subs,
            val_sub=val_sub,
            test_subs=test_subs,
            device=device,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            patience=patience,
            seed=seed,
            print_every=print_every
        )
    except TypeError:
        pass

    # Alternate names some versions used
    return train_one_fold_fn(
        X, y, sid,
        fold_train_subjects=train_subs,
        val_subject=val_sub,
        fold_test_subjects=test_subs,
        device=device,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        patience=patience,
        seed=seed,
        print_every=print_every
    )

def run_loso_kazemi_deterministic(dataset_name, X, y, sid, splits, device,
                                 epochs=60, steps_per_epoch=120, batch_size=64,
                                 lr=5e-4, weight_decay=1e-4, patience=10, seed=123, print_every=1):
    folds = list(iter_folds_from_splits_anyformat(splits, dataset_name))
    all_rows = []
    subj_rows = []

    for fold_i, fold in enumerate(folds, start=1):
        test_sub = int(fold["test"][0])
        train_full = [int(s) for s in fold["train"]]

        val_sub = pick_val_subject_deterministic(train_full)
        train_subs = [s for s in train_full if s != val_sub]
        test_subs = [test_sub]

        print(f"\n[KAZEMI STRONG] {dataset_name} fold {fold_i}/{len(folds)} | test={test_sub} | val={val_sub} | train_n={len(train_subs)}")
        print(f"settings: epochs={epochs}, steps/epoch={steps_per_epoch}, batch={batch_size}, lr={lr}, wd={weight_decay}, patience={patience}")

        t0 = time.time()
        out = _call_kazemi_train_one_fold(
            train_one_fold, X, y, sid,
            train_subs=train_subs, val_sub=val_sub, test_subs=test_subs,
            device=device, epochs=epochs, steps_per_epoch=steps_per_epoch,
            batch_size=batch_size, lr=lr, weight_decay=weight_decay,
            patience=patience, seed=seed + fold_i, print_every=print_every
        )
        sec = time.time() - t0

        # unpack
        if isinstance(out, tuple) and len(out) == 2:
            df_test, met = out
        else:
            raise ValueError("Unexpected return from train_one_fold. Expected (df_test, metrics_dict).")

        met = dict(met)
        met.update({"dataset": dataset_name, "fold": fold_i, "subject_id": test_sub, "val_subject": val_sub, "seconds": sec})

        df_test = df_test.copy()
        # enforce columns for saving/combining
        if "subject_id" not in df_test.columns:
            df_test["subject_id"] = test_sub
        df_test["dataset"] = dataset_name
        df_test["fold"] = fold_i
        df_test["val_subject"] = val_sub

        all_rows.append(df_test)
        subj_rows.append(met)

        print(f"[fold {fold_i}] done in {sec:.1f}s | metrics:", {k: met[k] for k in ["MAE","RMSE","N","best_val_mae","val_subject","dataset","fold","subject_id","seconds"] if k in met})

    df_all = pd.concat(all_rows, ignore_index=True)
    df_sub = pd.DataFrame(subj_rows)
    return df_all, df_sub


In [22]:
# --- sanity: print actual IDs, verify no leakage ---

def iter_folds_from_splits(splits: dict, dataset_name: str):
    """
    Supports your splits.json keys:
      - folds_WESAD
      - folds_PPG_DaLiA

    Supports fold dict formats:
      - {"train_subjects":[...], "test_subjects":[...]}
      - {"train":[...], "test":[...]}
    """
    ds = dataset_name.strip().lower()
    if ds.startswith("wesad"):
        key = "folds_WESAD"
    else:
        key = "folds_PPG_DaLiA"

    if key not in splits:
        raise KeyError(f"Missing '{key}' in splits. Available keys: {list(splits.keys())}")

    folds_obj = splits[key]
    if not isinstance(folds_obj, list) or len(folds_obj) == 0:
        raise ValueError(f"'{key}' must be a non-empty list. Got: {type(folds_obj)}")

    for fold in folds_obj:
        if not isinstance(fold, dict):
            raise ValueError(f"Each fold must be a dict. Got: {type(fold)}")

        if "train_subjects" in fold and "test_subjects" in fold:
            train = [int(s) for s in fold["train_subjects"]]
            test  = [int(s) for s in fold["test_subjects"]]
        elif "train" in fold and "test" in fold:
            train = [int(s) for s in fold["train"]]
            test  = [int(s) for s in fold["test"]]
        else:
            raise ValueError(f"Bad fold keys: {fold.keys()}")

        yield {"train": train, "test": test}


# pick fold 1
fold = next(iter_folds_from_splits(splits, "WESAD"))
train_subs = fold["train"]
test_subs  = fold["test"]

# IMPORTANT: set this to the val subject you are using in this fold
VAL_SUB = 3

train_wo_val = [s for s in train_subs if s != VAL_SUB]

print("train_subs:", sorted(train_subs))
print("val_sub:", VAL_SUB)
print("train_wo_val:", sorted(train_wo_val))
print("test_subs:", test_subs)

assert VAL_SUB not in test_subs, "VAL_SUB equals TEST subject (leakage)"
assert set(train_wo_val).isdisjoint(test_subs), "Train and test overlap (leakage)"
assert VAL_SUB in train_subs, "VAL_SUB not in training subjects for this fold"

print("✅ split sanity checks passed")


train_subs: [3, 4, 5, 6, 7, 8, 9, 10, 11]
val_sub: 3
train_wo_val: [4, 5, 6, 7, 8, 9, 10, 11]
test_subs: [2]
✅ split sanity checks passed


In [23]:
# Window-level sanity: make sure subject IDs match fold split
sid = np.asarray(wesad_sid).astype(int)

train_idx = np.where(np.isin(sid, train_wo_val))[0]
val_idx   = np.where(sid == VAL_SUB)[0]
test_idx  = np.where(np.isin(sid, test_subs))[0]

print("train windows:", len(train_idx))
print("val windows:", len(val_idx))
print("test windows:", len(test_idx))

assert len(test_idx) > 0, "No test windows found for test subject"
assert set(np.unique(sid[test_idx])).issubset(set(test_subs)), "Test windows contain non-test subjects"
assert set(np.unique(sid[val_idx])) == {VAL_SUB}, "Val windows contain non-val subjects"
assert set(np.unique(sid[train_idx])).issubset(set(train_wo_val)), "Train windows contain non-train subjects"

print("✅ window-level sanity passed")


train windows: 1414
val windows: 197
test windows: 186
✅ window-level sanity passed


## full solo

In [32]:
# Example: run both datasets (fusion arrays)
kaz_wesad_all, kaz_wesad_sub = run_loso_kazemi_deterministic(
    "WESAD", wesad_X_fusion, wesad_rr, wesad_sid,
    splits=splits, device=device,
    epochs=60, steps_per_epoch=120, batch_size=64,
    lr=5e-4, weight_decay=1e-4, patience=10, seed=123
)

kaz_dalia_all, kaz_dalia_sub = run_loso_kazemi_deterministic(
    "PPG-DaLiA", dalia_X_fusion, dalia_rr, dalia_sid,
    splits=splits, device=device,
    epochs=60, steps_per_epoch=120, batch_size=64,
    lr=5e-4, weight_decay=1e-4, patience=10, seed=123
)



[KAZEMI STRONG] WESAD fold 1/10 | test=2 | val=3 | train_n=8
settings: epochs=60, steps/epoch=120, batch=64, lr=0.0005, wd=0.0001, patience=10
train_one_fold: train_subs=8 val_sub=1 test_subs=1
  epoch 1/60 | val_mae=3.722 (val_subject=3)
  epoch 2/60 | val_mae=2.488 (val_subject=3)
  epoch 3/60 | val_mae=3.023 (val_subject=3)
  epoch 4/60 | val_mae=2.398 (val_subject=3)
  epoch 5/60 | val_mae=2.979 (val_subject=3)
  epoch 6/60 | val_mae=2.729 (val_subject=3)
  epoch 7/60 | val_mae=2.642 (val_subject=3)
  epoch 8/60 | val_mae=2.623 (val_subject=3)
  epoch 9/60 | val_mae=2.507 (val_subject=3)
  epoch 10/60 | val_mae=2.618 (val_subject=3)
  epoch 11/60 | val_mae=2.774 (val_subject=3)
  epoch 12/60 | val_mae=2.524 (val_subject=3)
  epoch 13/60 | val_mae=2.671 (val_subject=3)
  epoch 14/60 | val_mae=2.838 (val_subject=3)
[fold 1] done in 743.6s | metrics: {'MAE': 5.345210998289047, 'RMSE': 6.1834422487723915, 'N': 186, 'best_val_mae': 2.3984899520874023, 'val_subject': 3, 'dataset': 'WESA

In [30]:
from pathlib import Path
import os

root = Path.home() / "rr_runs"
print("RUNS_ROOT:", root)
print("exists:", root.exists())

if root.exists():
    # show latest 30 folders under rr_runs
    paths = sorted([p for p in root.rglob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)[:30]
    for p in paths:
        print(p)


RUNS_ROOT: C:\Users\yasmi\rr_runs
exists: True
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_133744
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_124555
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_124526
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_222540_ppg_only
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_222540_ppg_only
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD
C:\Users\yasmi\rr_runs\_tables
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_212700_ppg_only
C:\Users\yasmi\rr_runs\pimentel_baseline
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_212659_ppg_only
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_201212
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_195354
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_195331
C:\Users\yasmi\rr_runs\proposed_v6_tuned
C:\Users\yasmi\rr_run

In [31]:
from pathlib import Path

root = Path.home() / "rr_runs"
targets = ["per_window.csv", "per_subject.csv", "per_window_partial.csv", "per_subject_partial.csv", "last_fold.json"]

found = []
for t in targets:
    for p in root.rglob(t):
        found.append(p)

found = sorted(found, key=lambda p: p.stat().st_mtime, reverse=True)

print("Found:", len(found))
for p in found[:30]:
    print(p, "|", p.stat().st_size, "bytes")


Found: 14
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_222540_ppg_only\per_subject.csv | 654 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_222540_ppg_only\per_window.csv | 264740 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_222540_ppg_only\per_subject.csv | 463 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_222540_ppg_only\per_window.csv | 113309 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_212700_ppg_only\per_subject.csv | 906 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_212700_ppg_only\per_window.csv | 264740 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_212659_ppg_only\per_subject.csv | 649 bytes
C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_212659_ppg_only\per_window.csv | 113309 bytes
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_201212\per_subject.csv | 66 bytes
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_201212\per_window.cs

In [56]:
print("=== Name check ===")
for name in [
    "dalia_X", "dalia_X_fusion", "dalia_rr", "dalia_sid",
    "dalia_sqi_raw", "dalia_motion",
    "splits", "device"
]:
    print(name, "->", "OK" if name in globals() else "MISSING")

# also show shapes if possible
if "dalia_X" in globals():
    print("dalia_X shape:", dalia_X.shape)
if "dalia_X_fusion" in globals():
    print("dalia_X_fusion shape:", dalia_X_fusion.shape)
if "dalia_rr" in globals():
    print("dalia_rr shape:", dalia_rr.shape)
if "dalia_sid" in globals():
    print("dalia_sid shape:", dalia_sid.shape)


=== Name check ===
dalia_X -> MISSING
dalia_X_fusion -> OK
dalia_rr -> OK
dalia_sid -> OK
dalia_sqi_raw -> OK
dalia_motion -> MISSING
splits -> OK
device -> OK
dalia_X_fusion shape: (3883, 3, 2048)
dalia_rr shape: (3883,)
dalia_sid shape: (3883,)


In [57]:
import numpy as np

def find_fold_for_test_subject(splits, dataset_name, test_subject_id):
    folds = list(iter_folds_from_splits(splits, dataset_name))
    for i, f in enumerate(folds, start=1):
        test_subs = f["test"]
        if isinstance(test_subs, list) and len(test_subs) == 1 and int(test_subs[0]) == int(test_subject_id):
            return i, f, folds
    raise ValueError(f"No fold found for {dataset_name} with test subject {test_subject_id}")

# ---- choose which X to use ----
if "dalia_X_fusion" in globals():
    X_DALIA = dalia_X_fusion
elif "dalia_X" in globals():
    X_DALIA = dalia_X
else:
    raise NameError("No DaLiA X found. I expected dalia_X_fusion or dalia_X.")

SEED = 123

fold_i, fold, folds = find_fold_for_test_subject(splits, "PPG-DaLiA", 1)
train_subs_full = [int(s) for s in fold["train"]]

# deterministic val subject selection (matches what we did before)
rng = np.random.RandomState(SEED + 1337 + (fold_i - 1))
val_sub = int(rng.choice(train_subs_full))
train_subs = [s for s in train_subs_full if s != val_sub]
test_subs = [int(fold["test"][0])]

print(f"[KAZEMI STRONG] DaLiA fold {fold_i}/{len(folds)} | test={test_subs[0]} | val={val_sub} | train_n={len(train_subs)}")

df_test_kazemi, metrics_kazemi = train_one_fold(
    X_DALIA, dalia_rr, dalia_sid,
    fold_train_subjects=train_subs,
    fold_test_subjects=test_subs,
    device=device,
    val_subject=val_sub,
    epochs=60,
    steps_per_epoch=120,
    batch_size=64,
    lr=5e-4,
    patience=10,
    weight_decay=1e-4,
    max_train_windows=None,
    max_test_windows=None,
    print_every=1
)

print("\n[KAZEMI STRONG DaLiA RESULT]")
print(metrics_kazemi)
display(df_test_kazemi.head())


[KAZEMI STRONG] DaLiA fold 1/14 | test=1 | val=15 | train_n=12
train_one_fold: train_subs=12 val_sub=1 test_subs=1
  epoch 1/60 | val_mae=3.416 (val_subject=15)
  epoch 2/60 | val_mae=3.644 (val_subject=15)
  epoch 3/60 | val_mae=3.876 (val_subject=15)
  epoch 4/60 | val_mae=3.469 (val_subject=15)
  epoch 5/60 | val_mae=3.493 (val_subject=15)
  epoch 6/60 | val_mae=3.836 (val_subject=15)
  epoch 7/60 | val_mae=3.502 (val_subject=15)
  epoch 8/60 | val_mae=3.646 (val_subject=15)
  epoch 9/60 | val_mae=3.535 (val_subject=15)
  epoch 10/60 | val_mae=3.548 (val_subject=15)
  epoch 11/60 | val_mae=3.591 (val_subject=15)

[KAZEMI STRONG DaLiA RESULT]
{'MAE': 2.7428874174753823, 'RMSE': 3.5731440698597807, 'N': 288, 'best_val_mae': 3.4158201217651367, 'val_subject': 15}


,window_index,subject_id,rr_ref,rr_pred,is_valid
0,0,1,21.052631,17.170147,True
1,1,1,22.018349,13.395686,True
2,2,1,14.358974,15.792220,True
3,3,1,11.764706,15.195886,True
4,4,1,14.608696,15.087405,True


In [58]:
import numpy as np
import pandas as pd

def motion_feature_from_fusion(X):
    # Assumes channels: [PPG, ACC, GYRO] like your fusion tensor
    a = X[:, 1, :]
    g = X[:, 2, :]
    return np.sqrt(np.mean(a*a, axis=1) + np.mean(g*g, axis=1)).astype(np.float32)

def eval_strat_and_coverage_from_df(df, motion_all, sqi_all, dataset_name="PPG-DaLiA",
                                   cover_thresholds=(0.10,0.20,0.30,0.40,0.50)):
    df = df.copy()
    idx = df["window_index"].astype(int).to_numpy()

    df["motion"] = motion_all[idx]
    df["sqi01"]  = sqi_all[idx]
    df["abs_err"] = np.abs(df["rr_pred"].to_numpy() - df["rr_ref"].to_numpy())

    q1, q2 = np.quantile(df["motion"].to_numpy(), [1/3, 2/3])
    df["motion_bin"] = np.where(df["motion"] <= q1, "low",
                         np.where(df["motion"] <= q2, "mid", "high"))

    rows = []
    for b in ["all", "low", "mid", "high"]:
        d = df if b == "all" else df[df["motion_bin"] == b]
        mae = float(d["abs_err"].mean())
        rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
        within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
        rows.append([dataset_name, b, int(len(d)), mae, rmse, within3])

    strat = pd.DataFrame(rows, columns=["dataset","motion_bin","N","MAE","RMSE","pct_|err|<=3"])

    cov_rows = []
    for t in cover_thresholds:
        keep = df["sqi01"].to_numpy() >= float(t)
        cov = float(np.mean(keep))
        if keep.sum() == 0:
            cov_rows.append([dataset_name, t, cov, np.nan, np.nan, np.nan])
        else:
            d = df[keep]
            mae = float(d["abs_err"].mean())
            rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
            within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
            cov_rows.append([dataset_name, t, cov, mae, rmse, within3])

    cov = pd.DataFrame(cov_rows, columns=["dataset","sqi_threshold","coverage","MAE","RMSE","pct_|err|<=3"])
    return strat, cov

# ---- build motion if missing ----
if "dalia_motion" not in globals():
    # need fused X with ACC+GYRO channels
    if "dalia_X_fusion" in globals():
        dalia_motion = motion_feature_from_fusion(dalia_X_fusion)
    elif "dalia_X" in globals() and dalia_X.shape[1] >= 3:
        dalia_motion = motion_feature_from_fusion(dalia_X)
    else:
        raise NameError("No suitable DaLiA fusion tensor to compute motion. Need channels [PPG, ACC, GYRO].")

# ---- SQI must exist ----
if "dalia_sqi_raw" not in globals():
    raise NameError("dalia_sqi_raw is missing in this notebook. If you computed SQI in proposed notebook only, paste that SQI cell here too.")

strat_kaz, cov_kaz = eval_strat_and_coverage_from_df(
    df_test_kazemi,
    motion_all=dalia_motion,
    sqi_all=dalia_sqi_raw,
    dataset_name="PPG-DaLiA"
)

print("\n=== Kazemi STRONG DaLiA Motion-stratified ===")
display(strat_kaz)

print("\n=== Kazemi STRONG DaLiA Coverage tradeoff ===")
display(cov_kaz)



=== Kazemi STRONG DaLiA Motion-stratified ===


,dataset,motion_bin,N,MAE,RMSE,pct_|err|<=3
0,PPG-DaLiA,all,288,2.742887,3.573144,0.645833
1,PPG-DaLiA,low,96,3.038727,3.943392,0.583333
2,PPG-DaLiA,mid,96,2.790730,3.733410,0.666667
3,PPG-DaLiA,high,96,2.399205,2.968735,0.687500



=== Kazemi STRONG DaLiA Coverage tradeoff ===


,dataset,sqi_threshold,coverage,MAE,RMSE,pct_|err|<=3
0,PPG-DaLiA,0.1,0.503472,3.078072,4.003412,0.586207
1,PPG-DaLiA,0.2,0.322917,3.209999,4.228796,0.569892
2,PPG-DaLiA,0.3,0.229167,3.081561,3.986737,0.575758
3,PPG-DaLiA,0.4,0.187500,3.035035,4.043726,0.611111
4,PPG-DaLiA,0.5,0.138889,3.212408,4.355445,0.600000


NameError: Need dalia_X for proposed v6.

In [52]:
# =========================
# KAZEMI STRONG: single fold rerun (WESAD test subject 2)
# =========================

def find_fold_for_test_subject(splits, dataset_name, test_subject_id):
    folds = list(iter_folds_from_splits(splits, dataset_name))
    for i, f in enumerate(folds, start=1):
        test_subs = f["test"]
        if isinstance(test_subs, list) and len(test_subs) == 1 and int(test_subs[0]) == int(test_subject_id):
            return i, f, folds
    raise ValueError(f"No fold found for {dataset_name} with test subject {test_subject_id}")

# strong hyperparams (Option 1A+B)
EPOCHS = 60
STEPS_PER_EPOCH = 120
BATCH_SIZE = 64
LR = 5e-4
PATIENCE = 10
WEIGHT_DECAY = 1e-4
SEED = 123

fold_i, fold, folds = find_fold_for_test_subject(splits, "WESAD", 2)

# deterministically choose validation subject from training subjects
rng = np.random.RandomState(SEED + 1337 + (fold_i - 1))
val_subject = int(rng.choice([int(s) for s in fold["train"]]))

print(f"\n[KAZEMI STRONG] WESAD fold {fold_i}/{len(folds)} | test={fold['test'][0]} | val={val_subject} | train_n={len(fold['train'])-1}", flush=True)

df_test, metrics = train_one_fold(
    wesad_X_fusion, wesad_rr, wesad_sid,
    fold_train_subjects=fold["train"],     # train_one_fold will remove val internally
    fold_test_subjects=fold["test"],
    device=device,
    val_subject=val_subject,
    seed=SEED,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    weight_decay=WEIGHT_DECAY,
    print_every=1
)

print("\n[KAZEMI STRONG RESULT]")
print(metrics)
display(df_test.head())



[KAZEMI STRONG] WESAD fold 1/10 | test=2 | val=5 | train_n=8
train_one_fold: train_subs=8 val_sub=1 test_subs=1
  epoch 1/60 | val_mae=2.891 (val_subject=5)
  epoch 2/60 | val_mae=3.082 (val_subject=5)
  epoch 3/60 | val_mae=3.184 (val_subject=5)
  epoch 4/60 | val_mae=3.039 (val_subject=5)
  epoch 5/60 | val_mae=3.076 (val_subject=5)
  epoch 6/60 | val_mae=3.058 (val_subject=5)
  epoch 7/60 | val_mae=3.060 (val_subject=5)
  epoch 8/60 | val_mae=3.024 (val_subject=5)
  epoch 9/60 | val_mae=3.078 (val_subject=5)
  epoch 10/60 | val_mae=2.932 (val_subject=5)
  epoch 11/60 | val_mae=3.022 (val_subject=5)

[KAZEMI STRONG RESULT]
{'MAE': 4.519193613401023, 'RMSE': 5.384627721671087, 'N': 186, 'best_val_mae': 2.8911361694335938, 'val_subject': 5}


,window_index,subject_id,rr_ref,rr_pred,is_valid
0,0,2,16.153847,16.797714,True
1,1,2,22.641510,19.164127,True
2,2,2,24.444445,16.859230,True
3,3,2,24.000000,17.264114,True
4,4,2,24.220184,16.107752,True


In [55]:
# =========================
# Coverage + Motion-stratified evaluation for the single test subject run
# =========================

def motion_feature_from_fusion(X):
    a = X[:, 1, :]
    g = X[:, 2, :]
    return np.sqrt(np.mean(a*a, axis=1) + np.mean(g*g, axis=1)).astype(np.float32)

# build motion if you don't already have it
if "wesad_motion" not in globals():
    wesad_motion = motion_feature_from_fusion(wesad_X_fusion)

def eval_strat_and_coverage_from_df(df, motion_all, sqi_all, dataset_name="WESAD",
                                   cover_thresholds=(0.10,0.20,0.30,0.40,0.50)):
    df = df.copy()
    idx = df["window_index"].astype(int).to_numpy()
    df["motion"] = motion_all[idx]
    df["sqi01"] = sqi_all[idx]
    df["abs_err"] = np.abs(df["rr_pred"].to_numpy() - df["rr_ref"].to_numpy())

    # --- motion tertiles (low/mid/high) using THIS subject's windows
    q1, q2 = np.quantile(df["motion"].to_numpy(), [1/3, 2/3])
    df["motion_bin"] = np.where(df["motion"] <= q1, "low",
                         np.where(df["motion"] <= q2, "mid", "high"))

    # --- stratified summary
    rows = []
    for b in ["all", "low", "mid", "high"]:
        d = df if b == "all" else df[df["motion_bin"] == b]
        mae = float(d["abs_err"].mean())
        rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
        within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
        rows.append([dataset_name, b, int(len(d)), mae, rmse, within3])

    strat = pd.DataFrame(rows, columns=["dataset","motion_bin","N","MAE","RMSE","pct_|err|<=3"])

    # --- coverage table: abstain when SQI below threshold
    cov_rows = []
    for t in cover_thresholds:
        keep = df["sqi01"].to_numpy() >= float(t)
        cov = float(np.mean(keep))
        if keep.sum() == 0:
            cov_rows.append([dataset_name, t, cov, np.nan, np.nan, np.nan])
        else:
            d = df[keep]
            mae = float(d["abs_err"].mean())
            rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
            within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
            cov_rows.append([dataset_name, t, cov, mae, rmse, within3])

    cov = pd.DataFrame(cov_rows, columns=["dataset","sqi_threshold","coverage","MAE","RMSE","pct_|err|<=3"])
    return strat, cov

strat_df, cov_df = eval_strat_and_coverage_from_df(
    df_test,
    motion_all=wesad_motion,
    sqi_all=wesad_sqi_raw,
    dataset_name="WESAD"
)

print("\n=== Motion-stratified ===")
display(strat_df)

print("\n=== Coverage tradeoff (abstain if SQI < threshold) ===")
display(cov_df)



=== Motion-stratified ===


,dataset,motion_bin,N,MAE,RMSE,pct_|err|<=3
0,WESAD,all,186,4.519194,5.384628,0.354839
1,WESAD,low,62,4.868436,5.622233,0.338710
2,WESAD,mid,62,4.854527,5.783520,0.306452
3,WESAD,high,62,3.834618,4.682311,0.419355



=== Coverage tradeoff (abstain if SQI < threshold) ===


,dataset,sqi_threshold,coverage,MAE,RMSE,pct_|err|<=3
0,WESAD,0.1,0.940860,4.515894,5.402038,0.360000
1,WESAD,0.2,0.876344,4.526081,5.419890,0.355828
2,WESAD,0.3,0.811828,4.494111,5.403764,0.370861
3,WESAD,0.4,0.763441,4.465974,5.406755,0.380282
4,WESAD,0.5,0.736559,4.465762,5.404554,0.379562


In [65]:
# =========================
# SAVE KAZEMI OUTPUTS (replace old kazemi_outputs saving cell)
# =========================

# You must already have:
# wesad_all, wesad_sub, dalia_all, dalia_sub
# plus your hyperparams (EPOCHS, STEPS_PER_EPOCH, BATCH, LR, PATIENCE, WEIGHT_DECAY if you use it)

kazemi_config = {
    "method": "kazemi_strong",
    "epochs": int(EPOCHS) if "EPOCHS" in globals() else None,
    "steps_per_epoch": int(STEPS_PER_EPOCH) if "STEPS_PER_EPOCH" in globals() else None,
    "batch_size": int(BATCH) if "BATCH" in globals() else None,
    "lr": float(LR) if "LR" in globals() else None,
    "patience": int(PATIENCE) if "PATIENCE" in globals() else None,
    "weight_decay": float(WEIGHT_DECAY) if "WEIGHT_DECAY" in globals() else None,
    "notes": "Kazemi fusion baseline, strong training settings + subject-wise val",
}

# Save WESAD
run_dir_w = make_run_dir("kazemi_strong", "WESAD")
save_run(run_dir_w, wesad_all, wesad_sub, kazemi_config, splits_path="splits_loso.json")

# Save DaLiA
run_dir_d = make_run_dir("kazemi_strong", "PPG-DaLiA")
save_run(run_dir_d, dalia_all, dalia_sub, kazemi_config, splits_path="splits_loso.json")



✅ Saved run to: C:\Users\yasmi\rr_runs\kazemi_strong\WESAD\20260126_195201
  - per_window.csv | rows: 1797
  - per_subject.csv | rows: 10
  - config.json
  - splits: splits_loso.json OK

✅ Saved run to: C:\Users\yasmi\rr_runs\kazemi_strong\PPG-DaLiA\20260126_195202
  - per_window.csv | rows: 3883
  - per_subject.csv | rows: 14
  - config.json
  - splits: splits_loso.json OK


In [ ]:
import numpy as np

def summarize(df, name):
    mae_mean = df["MAE"].mean()
    mae_med  = df["MAE"].median()
    rmse_mean = df["RMSE"].mean()
    rmse_med  = df["RMSE"].median()
    print(f"\n=== {name} across-subject summary ===")
    print("MAE mean:", mae_mean, "| median:", mae_med)
    print("RMSE mean:", rmse_mean, "| median:", rmse_med)

summarize(wesad_sub, "WESAD (Kazemi early fusion)")
summarize(dalia_sub, "PPG-DaLiA (Kazemi early fusion)")


In [ ]:
BASELINE_DIR = r"C:\Jupyter Files\baseline_outputs"  # change if needed

ppg_wesad_sub = pd.read_csv(os.path.join(BASELINE_DIR, "baseline_pimentel_WESAD_per_subject.csv"))
ppg_dalia_sub = pd.read_csv(os.path.join(BASELINE_DIR, "baseline_pimentel_PPGDalia_per_subject.csv"))

def join_compare(ppg_df, fusion_df, name):
    m = ppg_df.merge(fusion_df, on="subject_id", suffixes=("_ppg", "_fusion"))
    m["MAE_drop"]  = m["MAE_ppg"]  - m["MAE_fusion"]
    m["RMSE_drop"] = m["RMSE_ppg"] - m["RMSE_fusion"]
    print(f"\n=== {name}: per-subject improvement (PPG-only minus Fusion) ===")
    print("Avg MAE drop:", m["MAE_drop"].mean())
    print("Avg RMSE drop:", m["RMSE_drop"].mean())
    return m

cmp_wesad = join_compare(ppg_wesad_sub, wesad_sub, "WESAD")
cmp_dalia = join_compare(ppg_dalia_sub, dalia_sub, "PPG-DaLiA")


In [ ]:
import os, glob

# put the folder you THINK has your baseline outputs
BASELINE_DIR = r"C:\Jupyter Files\baseline_outputs"

print("BASELINE_DIR exists?", os.path.isdir(BASELINE_DIR))
print("Files in BASELINE_DIR:")
for f in sorted(glob.glob(os.path.join(BASELINE_DIR, "*.csv"))):
    print("  ", os.path.basename(f))


In [ ]:
import os

def sniff(path, n=64):
    with open(path, "rb") as f:
        b = f.read(n)
    print(os.path.basename(path), "->", b[:8])

sniff(r"C:\Jupyter Files\baseline_outputs\baseline_pimentel_WESAD_per_window.csv")


In [ ]:
# check typical RR scale
print("WESAD rr range:", float(wesad_rr.min()), float(wesad_rr.max()))
print("DaLiA rr range:", float(dalia_rr.min()), float(dalia_rr.max()))



In [ ]:
import json, platform, sys, os
from datetime import datetime

def pkg_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return None

RUN_INFO = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        "numpy": pkg_version("numpy"),
        "pandas": pkg_version("pandas"),
        "scipy": pkg_version("scipy"),
        "sklearn": pkg_version("sklearn"),
        "torch": pkg_version("torch"),
    },

    # ---- your fixed data assumptions ----
    "fs_hz": FS,
    "win_samples": WIN_SAMPLES,
    "win_sec": WIN_SAMPLES / FS,

    # ---- channel mapping you used ----
    "channels": {
        "WESAD_PPG_CH": WESAD_PPG_CH,
        "DALIA_PPG_CH": DALIA_PPG_CH,
        "WESAD_ACC_CHS": WESAD_ACC_CHS,
        "WESAD_GYR_CHS": WESAD_GYR_CHS,
        "DALIA_ACC_CHS": DALIA_ACC_CHS,
        "DALIA_GYR_CHS": DALIA_GYR_CHS,
    },

    # ---- training config you actually used in train_one_fold ----
    "train_config": {
        "epochs": 100,
        "steps_per_epoch": 60,
        "batch_size": 64,
        "lr": 1e-3,
        "patience": 10,
        "loss": "SmoothL1Loss",
        "optimizer": "Adam",
        "scheduler": "CosineAnnealingLR",
        "seed": 0,
    },

    # ---- splits file you used ----
    "splits_json": SPLITS_JSON,

    # optional: output folder
    "out_dir": OUT_DIR,
}

os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "baseline_config_and_env.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(RUN_INFO, f, indent=2)

print("Wrote:", out_path)


In [ ]:
import os
import pandas as pd

# ✅ CHANGE THESE TWO
BASELINE_DIR = r"C:\Jupyter Files\baseline_outputs"
FUSION_DIR   = r"C:\Users\yasmi\OneDrive\Документы\kazemi_fusion_out"

# filenames (your earlier names)
BASE_WESAD = "baseline_pimentel_WESAD_per_subject.csv"
BASE_DALIA = "baseline_pimentel_PPGDalia_per_subject.csv"

FUS_WESAD  = "kazemi2024_fusion_WESAD_per_subject.csv"
FUS_DALIA  = "kazemi2024_fusion_PPGDalia_per_subject.csv"

def load_csv(folder, fn):
    path = os.path.join(folder, fn)
    assert os.path.exists(path), f"Missing file: {path}"
    df = pd.read_csv(path)
    print("Loaded:", fn, "| shape:", df.shape)
    return df

ppg_wesad = load_csv(BASELINE_DIR, BASE_WESAD)
ppg_dalia = load_csv(BASELINE_DIR, BASE_DALIA)

fus_wesad = load_csv(FUSION_DIR, FUS_WESAD)
fus_dalia = load_csv(FUSION_DIR, FUS_DALIA)

# make sure subject_id is int
for df in (ppg_wesad, ppg_dalia, fus_wesad, fus_dalia):
    if "subject_id" in df.columns:
        df["subject_id"] = df["subject_id"].astype(int)


In [ ]:
def compare_per_subject(ppg_df, fus_df, dataset_name):
    # keep only needed cols
    p = ppg_df[["subject_id", "MAE", "RMSE", "coverage", "N"]].copy()
    f = fus_df[["subject_id", "MAE", "RMSE", "N"]].copy()

    p = p.rename(columns={"MAE":"MAE_ppg", "RMSE":"RMSE_ppg", "N":"N_ppg"})
    f = f.rename(columns={"MAE":"MAE_fus", "RMSE":"RMSE_fus", "N":"N_fus"})

    merged = p.merge(f, on="subject_id", how="inner")

    merged["MAE_drop"]  = merged["MAE_ppg"]  - merged["MAE_fus"]
    merged["RMSE_drop"] = merged["RMSE_ppg"] - merged["RMSE_fus"]

    print(f"\n=== {dataset_name} joined subjects ===", len(merged))
    print("Subjects:", merged["subject_id"].tolist())

    print(f"\n=== {dataset_name} average improvement (PPG-only minus Fusion) ===")
    print("Avg MAE drop:", merged["MAE_drop"].mean())
    print("Avg RMSE drop:", merged["RMSE_drop"].mean())

    return merged.sort_values("subject_id")

wesad_cmp = compare_per_subject(ppg_wesad, fus_wesad, "WESAD")
dalia_cmp = compare_per_subject(ppg_dalia, fus_dalia, "PPG-DaLiA")

display(wesad_cmp)
display(dalia_cmp)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_baseline_vs_fusion(merged, dataset_name):
    subs = merged["subject_id"].astype(int).to_numpy()
    x = np.arange(len(subs))
    width = 0.38

    # ---- MAE plot ----
    plt.figure()
    plt.bar(x - width/2, merged["MAE_ppg"], width, label="PPG-only baseline")
    plt.bar(x + width/2, merged["MAE_fus"], width, label="Kazemi fusion")
    plt.xticks(x, subs, rotation=0)
    plt.ylabel("MAE (breaths/min)")
    plt.title(f"{dataset_name}: MAE per subject")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---- RMSE plot ----
    plt.figure()
    plt.bar(x - width/2, merged["RMSE_ppg"], width, label="PPG-only baseline")
    plt.bar(x + width/2, merged["RMSE_fus"], width, label="Kazemi fusion")
    plt.xticks(x, subs, rotation=0)
    plt.ylabel("RMSE (breaths/min)")
    plt.title(f"{dataset_name}: RMSE per subject")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_baseline_vs_fusion(wesad_cmp, "WESAD")
plot_baseline_vs_fusion(dalia_cmp, "PPG-DaLiA")


In [ ]:
def plot_improvement(merged, dataset_name):
    subs = merged["subject_id"].astype(int).to_numpy()
    x = np.arange(len(subs))

    plt.figure()
    plt.bar(x, merged["MAE_drop"], label="MAE drop (PPG - Fusion)")
    plt.xticks(x, subs)
    plt.ylabel("MAE improvement (breaths/min)")
    plt.title(f"{dataset_name}: MAE improvement per subject")
    plt.axhline(0)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_improvement(wesad_cmp, "WESAD")
plot_improvement(dalia_cmp, "PPG-DaLiA")


In [1]:
# =========================
# KAZEMI STRONG — FULL LOSO + SAVE (WESAD + DaLiA)
# =========================
import os, json, shutil
from datetime import datetime
import pandas as pd

RUNS_ROOT = r"C:\Users\yasmi\rr_runs"
METHOD_NAME = "kazemi_strong"
os.makedirs(RUNS_ROOT, exist_ok=True)

# ---- Strong, fair settings (same ones you used) ----
EPOCHS = 60
STEPS_PER_EPOCH = 120
BATCH = 64
LR = 5e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
SEED = 123

def make_run_dir(method, dataset):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    d = os.path.join(RUNS_ROOT, method, dataset, ts)
    os.makedirs(d, exist_ok=True)
    return d

def save_run(run_dir, per_window_df, per_subject_df, config_dict, splits_path=None):
    per_window_df.to_csv(os.path.join(run_dir, "per_window.csv"), index=False)
    per_subject_df.to_csv(os.path.join(run_dir, "per_subject.csv"), index=False)
    with open(os.path.join(run_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2)
    if splits_path is not None and os.path.exists(splits_path):
        shutil.copy2(splits_path, os.path.join(run_dir, "splits_loso.json"))
    print(f"\n✅ Saved run to: {run_dir}")
    print(f"  - per_window.csv | rows: {len(per_window_df)}")
    print(f"  - per_subject.csv | rows: {len(per_subject_df)}")
    print("  - config.json")
    print("  - splits: splits_loso.json OK" if (splits_path and os.path.exists(splits_path)) else "  - splits: (not copied)")

print("Running FULL LOSO: KAZEMI STRONG ...")
print(f"settings: epochs={EPOCHS}, steps/epoch={STEPS_PER_EPOCH}, batch={BATCH}, lr={LR}, wd={WEIGHT_DECAY}, patience={PATIENCE}")

# NOTE: run_loso must call your train_one_fold internally.
wesad_all, wesad_sub = run_loso(
    "WESAD",
    wesad_X_fusion, wesad_rr, wesad_sid,
    splits=splits,
    device=device,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH,
    patience=PATIENCE,
    lr=LR,
    seed=SEED
)

dalia_all, dalia_sub = run_loso(
    "PPG-DaLiA",
    dalia_X_fusion, dalia_rr, dalia_sid,
    splits=splits,
    device=device,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH,
    patience=PATIENCE,
    lr=LR,
    seed=SEED
)

config = dict(
    method=METHOD_NAME,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    note="subject-wise validation (val subject drawn from train subjects each fold)"
)

run_dir_w = make_run_dir(METHOD_NAME, "WESAD")
save_run(run_dir_w, wesad_all, wesad_sub, config, splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None)

run_dir_d = make_run_dir(METHOD_NAME, "PPG-DaLiA")
save_run(run_dir_d, dalia_all, dalia_sub, config, splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None)


Running FULL LOSO: KAZEMI STRONG ...
settings: epochs=60, steps/epoch=120, batch=64, lr=0.0005, wd=0.0001, patience=10


NameError: name 'run_loso' is not defined

In [33]:
import sys, platform, numpy as np, pandas as pd
try:
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torchv = torch.__version__
except Exception:
    dev = "unknown"
    torchv = "unknown"

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torchv)
print("device:", dev)


python: 3.13.5
platform: Windows-11-10.0.26100-SP0
numpy: 2.1.3
pandas: 2.2.3
torch: 2.9.1+cpu
device: cpu


In [34]:
import json, os, numpy as np

print("has splits in memory:", "splits" in globals())
if "SPLITS_JSON" in globals():
    print("SPLITS_JSON:", SPLITS_JSON, "| exists:", os.path.exists(SPLITS_JSON))

# show splits keys if present
if "splits" in globals():
    print("splits keys:", list(splits.keys()))

# quick fold peek (works with your anyformat iterator)
def _peek(dataset):
    folds = list(iter_folds_from_splits_anyformat(splits, dataset))
    print("\n===", dataset, "===")
    print("n_folds:", len(folds))
    print("fold1 keys:", folds[0].keys())
    print("fold1 train:", folds[0]["train"])
    print("fold1 test :", folds[0]["test"])
    return folds

_ = _peek("WESAD")
_ = _peek("PPG-DaLiA")


has splits in memory: True
SPLITS_JSON: C:\Jupyter Files\baseline_outputs\splits_loso.json | exists: True
splits keys: ['protocol', 'WESAD_subjects', 'PPG_DaLiA_subjects', 'folds_WESAD', 'folds_PPG_DaLiA']

=== WESAD ===
n_folds: 10
fold1 keys: dict_keys(['train', 'test'])
fold1 train: [3, 4, 5, 6, 7, 8, 9, 10, 11]
fold1 test : [2]

=== PPG-DaLiA ===
n_folds: 14
fold1 keys: dict_keys(['train', 'test'])
fold1 train: [2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]
fold1 test : [1]


In [35]:
from pathlib import Path

RUNS_ROOT = Path.home() / "rr_runs"
print("RUNS_ROOT:", RUNS_ROOT, "| exists:", RUNS_ROOT.exists())

def scan(method, dataset):
    base = RUNS_ROOT / method / dataset
    print(f"\n=== scan: {method} / {dataset} ===")
    print("base exists:", base.exists(), "| path:", base)
    if not base.exists():
        return
    runs = sorted([p for p in base.iterdir() if p.is_dir()], reverse=True)
    print("n_runs:", len(runs))
    for p in runs[:10]:
        files = {f.name for f in p.iterdir() if f.is_file()}
        flags = []
        if "per_window.csv" in files and "per_subject.csv" in files: flags.append("FINAL")
        if "per_window_partial.csv" in files and "per_subject_partial.csv" in files: flags.append("PARTIAL")
        if "config.json" in files: flags.append("CFG")
        if "splits_loso.json" in files: flags.append("SPLITS_COPY")
        print(" -", p.name, "|", ",".join(flags) if flags else "(no csv)")

# EDIT these method names to match your folders exactly:
scan("kazemi_strong", "WESAD")
scan("kazemi_strong", "PPG-DaLiA")

scan("proposed_v6_tuned", "WESAD")
scan("proposed_v6_tuned", "PPG-DaLiA")

scan("pimentel_baseline", "WESAD")
scan("pimentel_baseline", "PPG-DaLiA")


RUNS_ROOT: C:\Users\yasmi\rr_runs | exists: True

=== scan: kazemi_strong / WESAD ===
base exists: True | path: C:\Users\yasmi\rr_runs\kazemi_strong\WESAD
n_runs: 1
 - 20260126_195201 | FINAL,CFG,SPLITS_COPY

=== scan: kazemi_strong / PPG-DaLiA ===
base exists: True | path: C:\Users\yasmi\rr_runs\kazemi_strong\PPG-DaLiA
n_runs: 1
 - 20260126_195202 | FINAL,CFG,SPLITS_COPY

=== scan: proposed_v6_tuned / WESAD ===
base exists: True | path: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD
n_runs: 9
 - 20260128_133616_ckpt_detval | (no csv)
 - 20260128_111913_ckpt_detval | PARTIAL
 - 20260128_105330_ckpt_detval | (no csv)
 - 20260127_133744 | (no csv)
 - 20260127_124555 | (no csv)
 - 20260127_124526 | (no csv)
 - 20260126_201212 | FINAL,CFG,SPLITS_COPY
 - 20260126_195354 | (no csv)
 - 20260126_195331 | (no csv)

=== scan: proposed_v6_tuned / PPG-DaLiA ===
base exists: True | path: C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA
n_runs: 1
 - 20260128_111913_ckpt_detval | PARTIAL

=== scan:

In [36]:
import os, json, shutil
import pandas as pd

SPLITS_JSON = r"C:\Jupyter Files\baseline_outputs\splits_loso.json"
assert os.path.exists(SPLITS_JSON), f"Missing SPLITS_JSON: {SPLITS_JSON}"

PROPOSED_WESAD_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval"
PROPOSED_DALIA_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval"

def _safe_write_csv(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def finalize_partials(run_dir):
    pw_part = os.path.join(run_dir, "per_window_partial.csv")
    ps_part = os.path.join(run_dir, "per_subject_partial.csv")
    pw_out  = os.path.join(run_dir, "per_window.csv")
    ps_out  = os.path.join(run_dir, "per_subject.csv")

    print("\n=== Finalize ===")
    print("run_dir:", run_dir)
    assert os.path.exists(run_dir), "Run dir does not exist"

    # Copy splits (optional but nice to keep consistent)
    dst_splits = os.path.join(run_dir, "splits_loso.json")
    if not os.path.exists(dst_splits):
        shutil.copy2(SPLITS_JSON, dst_splits)
        print("copied splits_loso.json ✅")
    else:
        print("splits_loso.json already exists ✅")

    # Finalize per_window
    if os.path.exists(pw_part):
        dfw = pd.read_csv(pw_part)
        # light de-dup if resume ever appended duplicates
        key_cols = [c for c in ["dataset","fold","subject_id","window_index"] if c in dfw.columns]
        if len(key_cols) >= 2:
            dfw = dfw.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfw, pw_out)
        print("wrote per_window.csv ✅ | rows:", len(dfw))
    elif os.path.exists(pw_out):
        dfw = pd.read_csv(pw_out)
        print("per_window.csv already exists ✅ | rows:", len(dfw))
    else:
        raise FileNotFoundError("Neither per_window_partial.csv nor per_window.csv exists")

    # Finalize per_subject
    if os.path.exists(ps_part):
        dfs = pd.read_csv(ps_part)
        key_cols = [c for c in ["dataset","fold","subject_id"] if c in dfs.columns]
        if len(key_cols) >= 2:
            dfs = dfs.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfs, ps_out)
        print("wrote per_subject.csv ✅ | rows:", len(dfs))
    elif os.path.exists(ps_out):
        dfs = pd.read_csv(ps_out)
        print("per_subject.csv already exists ✅ | rows:", len(dfs))
    else:
        raise FileNotFoundError("Neither per_subject_partial.csv nor per_subject.csv exists")

finalize_partials(PROPOSED_WESAD_DIR)
finalize_partials(PROPOSED_DALIA_DIR)

print("\n✅ Done. Proposed det-val runs are now FINAL.")



=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval
splits_loso.json already exists ✅
wrote per_window.csv ✅ | rows: 1797
wrote per_subject.csv ✅ | rows: 10

=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval
splits_loso.json already exists ✅
wrote per_window.csv ✅ | rows: 3883
wrote per_subject.csv ✅ | rows: 14

✅ Done. Proposed det-val runs are now FINAL.
